# 📊 Data Exploration & Overview

**UIDAI Aadhaar Insights Project**

This notebook provides an introduction to the UIDAI Aadhaar datasets with interactive exploration.

---

## 📌 Objectives
1. Load and understand the three core datasets
2. Assess data quality (missing values, duplicates)
3. Generate summary statistics
4. Create initial visualizations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Define paths
DATA_DIR = Path('../data/processed')

print('✅ Libraries loaded successfully!')

---

## 1️⃣ Load Datasets

We have three main datasets:
- **Enrolment**: New Aadhaar registrations
- **Demographic Updates**: Changes to demographic information
- **Biometric Updates**: Updates to biometric data

In [ ]:
# Load all three datasets
print('Loading datasets...')

enrolment = pd.read_csv(DATA_DIR / 'cleaned_enrolment.csv', parse_dates=['date'])
demographic = pd.read_csv(DATA_DIR / 'cleaned_demographic.csv', parse_dates=['date'])
biometric = pd.read_csv(DATA_DIR / 'cleaned_biometric.csv', parse_dates=['date'])

datasets = {
    'Enrolment': enrolment,
    'Demographic': demographic,
    'Biometric': biometric
}

print('✅ All datasets loaded!')

---

## 2️⃣ Dataset Overview

Let's understand the structure and size of each dataset.

In [ ]:
# Dataset shapes
print('=' * 60)
print('📊 DATASET DIMENSIONS')
print('=' * 60)

for name, df in datasets.items():
    print(f'\n{name}:')
    print(f'  📦 Rows: {df.shape[0]:,}')
    print(f'  📋 Columns: {df.shape[1]}')
    print(f'  💾 Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

In [ ]:
# Column information
print('=' * 60)
print('📋 COLUMN DETAILS')
print('=' * 60)

for name, df in datasets.items():
    print(f'\n🔹 {name} Dataset:')
    print(f'   Columns: {list(df.columns)}')

In [ ]:
# Sample data from each dataset
print('=' * 60)
print('👀 SAMPLE DATA - ENROLMENT')
print('=' * 60)
enrolment.head()

In [ ]:
print('=' * 60)
print('👀 SAMPLE DATA - DEMOGRAPHIC')
print('=' * 60)
demographic.head()

In [ ]:
print('=' * 60)
print('👀 SAMPLE DATA - BIOMETRIC')
print('=' * 60)
biometric.head()

---

## 3️⃣ Data Quality Assessment

Check for missing values, duplicates, and data types.

In [ ]:
# Missing values analysis
print('=' * 60)
print('🔍 MISSING VALUES ANALYSIS')
print('=' * 60)

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    print(f'\n🔹 {name}:')
    if missing.sum() == 0:
        print('   ✅ No missing values!')
    else:
        print(missing[missing > 0])

In [ ]:
# Data types
print('=' * 60)
print('📊 DATA TYPES')
print('=' * 60)

for name, df in datasets.items():
    print(f'\n🔹 {name}:')
    print(df.dtypes.to_string())

In [ ]:
# Date range for each dataset
print('=' * 60)
print('📅 DATE RANGES')
print('=' * 60)

for name, df in datasets.items():
    print(f'\n🔹 {name}:')
    print(f'   Start: {df["date"].min()}')
    print(f'   End: {df["date"].max()}')
    print(f'   Days: {(df["date"].max() - df["date"].min()).days}')

---

## 4️⃣ Summary Statistics

Generate descriptive statistics for numeric columns.

In [ ]:
# Enrolment statistics
print('📊 ENROLMENT SUMMARY STATISTICS')
enrolment.describe()

In [ ]:
# Demographic statistics
print('📊 DEMOGRAPHIC SUMMARY STATISTICS')
demographic.describe()

In [ ]:
# Biometric statistics
print('📊 BIOMETRIC SUMMARY STATISTICS')
biometric.describe()

---

## 5️⃣ Initial Visualizations

Quick visual overview of the data.

In [ ]:
# Create figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Total records per dataset
dataset_totals = [len(df) for df in datasets.values()]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

axes[0].bar(datasets.keys(), dataset_totals, color=colors)
axes[0].set_title('📊 Total Records per Dataset', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Records')
for i, v in enumerate(dataset_totals):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', fontsize=10)

# Unique states
unique_states = [df['state'].nunique() for df in datasets.values()]
axes[1].bar(datasets.keys(), unique_states, color=colors)
axes[1].set_title('🗺️ Unique States per Dataset', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of States')
for i, v in enumerate(unique_states):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontsize=10)

# Unique districts
unique_districts = [df['district'].nunique() for df in datasets.values()]
axes[2].bar(datasets.keys(), unique_districts, color=colors)
axes[2].set_title('📍 Unique Districts per Dataset', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Number of Districts')
for i, v in enumerate(unique_districts):
    axes[2].text(i, v + 5, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly trends overview
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (name, df) in enumerate(datasets.items()):
    # Identify numeric columns (age groups)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if 'pincode' not in c.lower()]
    
    # Monthly totals - FIXED VERSION
    df['month'] = df['date'].dt.to_period('M').astype(str)
    df['total'] = df[numeric_cols].sum(axis=1)
    monthly = df.groupby('month')['total'].sum()
    
    axes[idx].plot(monthly.index, monthly.values, 
                   marker='o', linewidth=2, markersize=6, color=colors[idx])
    axes[idx].set_title(f'📈 {name} - Monthly Trend', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Total Count')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6️⃣ Key Observations

### Summary:
- **Enrolment**: New Aadhaar registrations with age group breakdown (0-5, 5-17, 18+)
- **Demographic**: Updates to demographic info with same age breakdown
- **Biometric**: Biometric updates with age breakdown (5-17, 17+)

### Data Quality:
- All datasets have been pre-cleaned
- Date ranges span the analysis period
- Geographic coverage includes all states and union territories

### Next Steps:
- Proceed to **02_deep_dive_eda.ipynb** for detailed analysis
- Explore state-wise patterns and temporal trends

In [ ]:
print('\n' + '=' * 60)
print('✅ DATA EXPLORATION COMPLETE!')
print('=' * 60)
print('\n📌 Next: Open 02_deep_dive_eda.ipynb for detailed analysis')